In [ ]:
import torch
import torch.optim as optim
from torchviz import make_dot
import pandas as pd
from itables import init_notebook_mode, show
from sklearn.preprocessing import LabelEncoder, OneHotEncoder
from sklearn.model_selection import train_test_split
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import KFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, ConfusionMatrixDisplay
import importlib
from tqdm import tqdm

import deeparguing.gradual_aacbr as gradual_aacbr
import deeparguing.semantics.mlp_semantics as ms
import deeparguing.base_scores.feature_weighted_base_score as fwbs
import deeparguing.casebase_edge_weights.feature_weighted_partial_order as fwpo
import deeparguing.casebase_edge_weights.learned_partial_order as lpo
import deeparguing.irrelevance_edge_weights.feature_weighted_irrelevance as fwi
import deeparguing.irrelevance_edge_weights.regular_irrelevance as ri

init_notebook_mode(all_interactive=True)

In [ ]:
def reload_imports():
    importlib.reload(gradual_aacbr)
    importlib.reload(ms)
    importlib.reload(fwbs)
    importlib.reload(fwpo)
    importlib.reload(fwi)
    importlib.reload(ri)
    importlib.reload(lpo)

reload_imports()

## Data Set

In [ ]:
SEED = 42

In [ ]:
# from ucimlrepo import fetch_ucirepo 
  
# # fetch dataset 
# connectionist_bench_sonar_mines_vs_rocks = fetch_ucirepo(id=151) 
  
# # data (as pandas dataframes) 
# X = connectionist_bench_sonar_mines_vs_rocks.data.features 
# y = connectionist_bench_sonar_mines_vs_rocks.data.targets 

data = pd.read_csv('../data/iris/iris.data')

data = data.values

X = np.array(data[:, :-1], dtype=np.float32)
y = np.array(data[:, -1])

show(X)
print(np.unique(y))


In [ ]:
y = y.reshape(-1, 1)
encoder = OneHotEncoder(sparse_output=False)
# encoder = LabelEncoder()
encoder.fit(y)
y = encoder.transform(y)

show(y)

all_y = np.unique(y, axis=0)

## Train Model

### Split into Training, Validation and Test

In [ ]:
X, y = torch.tensor(X), torch.tensor(y, dtype=torch.float32)
X_train_full, X_test, y_train_full, y_test = train_test_split(X, y, test_size=0.2, random_state=SEED)
X_train, X_val, y_train, y_val = train_test_split(X_train_full, y_train_full, test_size=0.2, random_state=SEED)
print(f"Test Size:  {len(X_test)}")
print(f"Train Size:  {len(X_train)}")
print(f"Validation Size:  {len(X_val)}")

### Normalize dataset

In [ ]:
# train_mean = X_train.mean(dim=0)
# train_std = X_train.std(dim=0)

# X_train = (X_train - train_me(X_train, y_train, X_default, y_default, use_symmetric_attacks, defaults_not_attack)

In [ ]:
print(X_train)

### Build AF

In [50]:
means = X_train.mean(axis=0)
std = X_train.std(axis=0)

In [51]:
PREPROCESS_FUNC = lambda x: x

### Train Model

In [ ]:
DEFAULT_CASE = means

X_DEFAULTS = PREPROCESS_FUNC(DEFAULT_CASE.tile(len(all_y), 1))
Y_DEFAULTS = torch.tensor(all_y)

MAX_ITERS = 1 
EPOCHS = 75 
N_SPLITS = 5
USE_SYMMETRIC_ATTACKS = True

In [ ]:
reload_imports()

def run_gradual_model(model: gradual_aacbr.GradualAACBR, X_train, y_train, X_default, 
                      y_default, new_cases):

    model.fit(PREPROCESS_FUNC(X_train), y_train, X_default, y_default, USE_SYMMETRIC_ATTACKS)
    return model(PREPROCESS_FUNC(new_cases))


def evaluate_model(model, X_train, y_train, X_default, y_default, new_cases, new_cases_labels,
                   show_confusion=False, print_graph=False, print_matrix=False, print_compute_graph = False):


    final_strengths = run_gradual_model(model, X_train, y_train, X_default, y_default, new_cases)
    predicted = final_strengths.cpu().detach().numpy()

    predicted = np.argmax(predicted, axis=1)
    new_cases_labels_orig = np.argmax(new_cases_labels, axis=1)

    print("Accuracy, Precision, Recall, F1")
    print([
        accuracy_score(new_cases_labels_orig, predicted),
        precision_score(new_cases_labels_orig, predicted, average='macro'),
        recall_score(new_cases_labels_orig, predicted, average='macro'),
        f1_score(new_cases_labels_orig, predicted, average='macro')
    ])


    if show_confusion:
        cm = confusion_matrix(new_cases_labels_orig, predicted)
        disp = ConfusionMatrixDisplay(confusion_matrix=cm)
        disp.plot()
        plt.show()

    if print_graph:
        model.show_graph_with_labels()

    if print_matrix:
        model.show_matrix()

    if print_compute_graph:
        criterion = torch.nn.CrossEntropyLoss()
        loss = criterion(final_strengths.squeeze(), new_cases_labels)
        make_dot(loss, params=dict(model.named_parameters())).render("gradual_aacbr", format="pdf")



In [ ]:
reload_imports()
torch.manual_seed(0) # TRY DIFFERENT INITIAL WEIGHTS 
kf = KFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)

no_features = X_train.shape[-1]
semantics = ms.MLPBasedSemantics(max_iters=MAX_ITERS, epsilon=0)
partial_order = fwpo.FeatureWeightedPartialOrder(no_features, sharpness=100)
# irrelevance = ri.RegularIrrelevance(partial_order)
irrelevance = fwi.FeatureWeightedIrrelevance(no_features)
base_score = fwbs.FeatureWeightedBaseScore(no_features)

model = gradual_aacbr.GradualAACBR(semantics, 
                                base_score,
                                irrelevance,
                                partial_order)

# criterion = torch.nn.BCELoss()
criterion = torch.nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=0.1, momentum=0.9)

In [ ]:
reload_imports()
evaluate_model(model, X_train, y_train, X_DEFAULTS, Y_DEFAULTS, X_val, y_val, show_confusion=True,  print_matrix=True, print_compute_graph=True, print_graph = False )
show(model.A.detach().numpy())
model.plot_casebase_edge_weights_parameters()

In [ ]:

DISABLE_TQDM = False

In [ ]:
reload_imports()
losses = []
pbar = tqdm(range(EPOCHS), disable=DISABLE_TQDM)

for epoch in pbar:  
    for fold, (casebase_index,  new_cases_index) in enumerate(kf.split(X_train)):

        new_cases = X_train[new_cases_index]
        new_cases_labels = y_train[new_cases_index]

        casebase = X_train[casebase_index]
        casebase_labels = y_train[casebase_index]

        optimizer.zero_grad()

        # TODO: consider efficiency issues with having to rebuild each time 
        # Find a way to accumulate gradients update only when necessary?
        model.fit(casebase, casebase_labels, X_DEFAULTS, Y_DEFAULTS, USE_SYMMETRIC_ATTACKS)

        predictions = model(new_cases).squeeze()
        loss = criterion(predictions, new_cases_labels)
        loss.backward()

        optimizer.step()

        losses.append(loss.item()/len(new_cases))
        # Print gradients
        # print("Gradients for each layer:")
        # for name, param in model.named_parameters():
        #     if param.grad is not None:
        #         print(f"Layer: {name} | Gradients: {param.grad}")
        #     else:
        #         print(f"Layer: {name} | No gradients computed")


    pbar.set_description(f'Epoch {epoch + 1}, Loss: {round(loss.item()/len(new_cases), 4)}')






print('Finished Training')

plt.plot(losses)
plt.show()


In [ ]:
reload_imports()
with torch.no_grad():
    evaluate_model(model, X_train, y_train, X_DEFAULTS, Y_DEFAULTS, X_val, y_val, show_confusion=True,  print_matrix=True, print_compute_graph=False  )

model.plot_casebase_edge_weights_parameters()

### Test Set

In [ ]:
# reload_imports()
# with torch.no_grad():
#     evaluate_model(X_train_full, y_train_full, X_test, y_test, show_confusion=True)